In [5]:
#1-Defining the class which is logs me in to the viewer website with more flexibility than fct used in other code
#CandidateViewerQuery and CandidateViewerRegistrar
import requests

class AutomationAPI:
    def __init__(self, base_url, survey_id, username, password):
        self.url = base_url
        self.survey_id = survey_id
        self.username = username
        self.password = password
        self.survey_aval = []
        self.session = requests.Session()

        # Login
        self._call("login", username=username, password=password)
        
        # Register endpoints
        self._register_endpoints()
        
        # Get list of surveys
        self.survey_aval = self._call("get_all_surveys")["surveys"]

        # Check if survey_id is valid
        if survey_id not in self.survey_aval:
            raise ValueError(f"Survey ID {survey_id} is not available. Available surveys: ", self.survey_aval)
        
        # Set survey
        self._call("set_survey", survey=survey_id)

    def _call(self, endpoint, **params):
        response = self.session.post(
            self.url,
            params={"endpoint": endpoint},  # GET
            data=params                      # POST
        )
        response.raise_for_status()
        
        try:
            data = response.json()
            if data["status"] != "success":
                raise RuntimeError(data["message"])
        except ValueError:
            raise RuntimeError("Invalid JSON response", response.text)
        
        return data["data"]

    def _register_endpoints(self):
        for endpoint in self._call("get_endpoints_aval")["endpoints"]:
            name = endpoint["name"]
            param_names = endpoint["params"]

            if hasattr(self, name):
                continue

            def make_method(endpoint_name, endpoint_params):
                def method(self, **kwargs):
                    missing = [p for p in endpoint_params if p not in kwargs]
                    if missing:
                        raise ValueError(f"Missing parameter(s): {', '.join(missing)}")
                    return self._call(endpoint_name, **kwargs)
                method.__name__ = endpoint_name
                method.__doc__  = f"Params: {', '.join(endpoint_params)}"
                return method

            setattr(self.__class__, name, make_method(name, param_names))

In [6]:
#2-Calling the function for the daily cands
api = AutomationAPI(
    "https://sps.chimenet.ca/candidates/index.php?automation", "dailycands", "Viewer Bot", "v4A13BNYwqU5okUZE^h9c&x*blzHrYMi"
)

new_cand_info = api.get_files_by_rating_type(folder="2026-04-07", rating_type="new_candidates")["files"]
print(new_cand_info)
print("-"*40) #just a line to separate candidate
faint_cand_info = api.get_files_by_rating_type(folder="2026-04-07", rating_type="faint")["files"]
print(faint_cand_info)
#tags for new cand already have some'Confirmed Pulsar', but for the faint ones it is empty most of the time

[{'file': 'Multi_Pointing_Groups_f_2.984_DM_75.291_69d8927fbe9b9f8ba6354728', 'folder': '2026-04-07', 'remoteUrl': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_2.984_DM_75.291_69d8927fbe9b9f8ba6354728&folder=2026-04-07&type=candidate_image', 'remoteUrl_alt': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_2.984_DM_75.291_69d8927fbe9b9f8ba6354728&folder=2026-04-07&type=candidate_image_alt', 'checked': {'status': True, 'result': 'NEW CANDIDATE', 'date': '1775867261', 'by': 'Viewer Bot', 'rater_results': {'Wenke Xia': {'result': 'NEW CANDIDATE', 'additional': False}, 'Reynier Squillace': {'result': 'NEW CANDIDATE', 'additional': False}}, 'rating_consistency': {'consistent': True, 'status': 'consistent', 'result': 'NEW CANDIDATE', 'date': '1775867261'}, 'info': {'history': []}, 'tags': [{'id': '90', 'survey': 'dailycands', 'folder': '2026-04-07', 'file': 'Multi_Pointing_Groups_f_2.984_DM_75.291_69d8927fbe9b9f8ba6354728', 'tag': 'Confirmed Pulsar', 'added_by': 'Wenk

In [7]:
#3-Fct to get tag(to see if it worked)

def extract_new_tags(new_cand_info):
    for f in new_cand_info:
        tags = f["checked"]["tags"]#to access the tag in the dictionnary that contains "files" which is a list where each element are itself a dictionnary

        if tags:
            print(tags[0]["tag"])
            
def extract_faint_tags(data):
    for f in data:
        print(f["checked"]["tags"]) 
extract_new_tags(new_cand_info)
print("-"*40)
extract_faint_tags(faint_cand_info)

Confirmed Pulsar
----------------------------------------
[]
[]
[]


In [9]:
#Example to then put in the query(main) loop
#Let's try a loop to add tag for faint one in test

#2-Calling the function for the daily cands
api = AutomationAPI(
    "https://sps.chimenet.ca/candidates/index.php?automation", "test", "Viewer Bot", "v4A13BNYwqU5okUZE^h9c&x*blzHrYMi"
)

faint_data = api.get_files_by_rating_type(folder="test_21", rating_type="faint")
print(faint_data)

{'files': [{'file': 'Multi_Pointing_Groups_f_28.890_DM_20.240_690b7d1b999bcd1baf29e75b', 'folder': 'test_21', 'remoteUrl': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_28.890_DM_20.240_690b7d1b999bcd1baf29e75b&folder=test_21&type=candidate_image', 'remoteUrl_alt': '/candidates/index.php?assets&file=Multi_Pointing_Groups_f_28.890_DM_20.240_690b7d1b999bcd1baf29e75b&folder=test_21&type=candidate_image_alt', 'checked': {'status': True, 'result': '<faint>', 'date': '1772653126', 'by': 'Viewer Bot', 'rater_results': {'Wenke Xia': {'result': '<none>', 'additional': False}, 'Viewer Admin': {'result': '<faint>', 'additional': False}}, 'rating_consistency': {'consistent': True, 'status': 'faint_consistent', 'result': '<faint>', 'date': '1772653126'}, 'info': {'history': []}, 'tags': [{'id': '132', 'survey': 'test', 'folder': 'test_21', 'file': 'Multi_Pointing_Groups_f_28.890_DM_20.240_690b7d1b999bcd1baf29e75b', 'tag': 'Test Tag', 'added_by': 'Viewer Bot', 'time': '1778080156', 'inf

In [10]:
for f in faint_data["files"]:
    api.add_tag(
        folder="test_21",
        file=f["file"],
        tag="Test Tag"
    )
    api.add_tag(
        folder="test_21",
        file=f["file"],
        tag="Test Tag2"
    )
    api.add_tag(
        folder="test_21",
        file=f["file"],
        tag="Test Tag3"
    )

# refresh data
updated = api.get_files_by_rating_type(folder="test_21", rating_type="faint")

for f in updated["files"]:
    for t in f["checked"]["tags"]:
        print(t["tag"])

#here we had 3 faint psr but one has been further classified has a confirmed pulsar
#so when we had the tag then we have a second name in the list

Test Tag2
Test Tag3
Test Tag
Confirmed Pulsar
Test Tag2
Test Tag3
Test Tag
Test Tag2
Test Tag3
Test Tag


In [11]:
#if want to remove it 
for f in faint_data["files"]:
    file_name = f["file"]

    api.delete_tag(
        folder="test_21",
        file=file_name,
        tag="Test Tag"
    )

# re-fetch updated data
updated = api.get_files_by_rating_type(folder="test_21", rating_type="faint")

# print once
extract_faint_tags(updated["files"])
#We only remove the tag of 'Confirmed Pulsar' so it leaves it unchanged

[{'id': '139', 'survey': 'test', 'folder': 'test_21', 'file': 'Multi_Pointing_Groups_f_28.890_DM_20.240_690b7d1b999bcd1baf29e75b', 'tag': 'Test Tag2', 'added_by': 'Viewer Bot', 'time': '1778088098', 'info': '[]', 'color': '#8BC34A', 'style': 'label-default', 'text': 'Test Tag2'}, {'id': '140', 'survey': 'test', 'folder': 'test_21', 'file': 'Multi_Pointing_Groups_f_28.890_DM_20.240_690b7d1b999bcd1baf29e75b', 'tag': 'Test Tag3', 'added_by': 'Viewer Bot', 'time': '1778088098', 'info': '[]', 'color': '#673AB7', 'style': 'label-default', 'text': 'Test Tag3'}, {'id': '29', 'survey': 'test', 'folder': 'test_21', 'file': 'Multi_Pointing_Groups_f_28.890_DM_20.240_690b7d1b999bcd1baf29e75b', 'tag': 'Confirmed Pulsar', 'added_by': 'Wenke Xia', 'time': '1775192918', 'info': '[]', 'color': '#5cb85c', 'style': 'label-default', 'text': 'Confirmed Pulsar'}]
[{'id': '141', 'survey': 'test', 'folder': 'test_21', 'file': 'Multi_Pointing_Groups_f_8.824_DM_35.014_69098381999bcd1baff08388', 'tag': 'Test Tag2

In [12]:
# refresh data
updated = api.get_files_by_rating_type(folder="test_21", rating_type="faint")

for f in updated["files"]:
    for t in f["checked"]["tags"]:
        print(t["tag"])
    

Test Tag2
Test Tag3
Confirmed Pulsar
Test Tag2
Test Tag3
Test Tag2
Test Tag3


In [ ]:
#4-api.get_created_tags() function to create a tag
#api.add_tag(folder="test_22", file="Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368", tag="Test Tag")
#api.get_file_details(folder="test_22", file="Multi_Pointing_Groups_f_7.746_DM_107.877_6907f554999bcd1bafd30368")